# C1b · Resta por exposición y combinado

**Spec:** [`docs/docs/spec_C1b_codex_perobs_subtraction.md`](../docs/docs/spec_C1b_codex_perobs_subtraction.md)  |  **Bloque:** C · Extracción  |  **Run de este set:** `ROXs42Bb_realigned`

Resta a cada exposición SU modelo de PSF y combina los residuos, en vez de restar un modelo único al cubo ya combinado.

| | |
|---|---|
| **Entrada** | `psf_model.json` de forma `mixture` (C1 con `psf_scope=per_observation`), `observation_plan.json` y los cubos por exposición |
| **Salida (QC/productos)** | `stages/cube_psfsub_perobs.fits` + `stages/stage_e01b_qc.json` |
| **Consume aguas abajo** | C3 (variante `psfsub`) |


## Qué hace C1b y por qué

Restar el halo de la primaria sobre el cubo **combinado** obliga a describir con un solo modelo la mezcla de 29–30 PSF distintas. Con la PSF ya ajustada por observación (C1), la resta se hace **donde el modelo vale** —en su propia exposición— y la combinación viene después.

Reutiliza el combinado que ya existe (`stream_combine.combine_streaming`) a través de su gancho `transform`: cada trozo se recorta y alinea igual que siempre, se le ajusta a su modelo la amplitud y el fondo por canal (la misma función que usa C3) y se resta. La memoria queda acotada por el trozo, no por el número de exposiciones.

**El fondo ajustado no se resta** (igual que en C3: lo que se quita es `amp·PSF`), y `STAT` no se toca, porque restar un modelo determinista no cambia la varianza.

**Rol en la cadena:** entrega `cube_psfsub_perobs.fits`, que es lo que consume la variante `psfsub` de C3. Las demás variantes siguen sobre el cubo combinado.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python -m musepipe.stages.stage_e01b_perobs_subtract --run-id $RUN
```

Pesado: una pasada de lectura sobre las 29–30 exposiciones.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_e01b_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -m musepipe.stages.stage_e01b_perobs_subtract --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_e01b_qc.json', RUN_ID)
nb.show(qc, keys=['input.n_components', 'combine.n_exposures', 'combine.rejected_fraction', 'subtraction.n_exposures', 'subtraction.amplitude_median_spread_pct'], title='C1b')


## Los términos de este QC, en físico

| Término | Qué es | Por qué importa |
|---|---|---|
| `input.n_components` | Cuántas exposiciones traía la mezcla de C1. | Tiene que ser el número de exposiciones que el plan del combinado usa: si falta alguna, el cubo residual no es el mismo campo que el combinado. |
| `subtraction.per_exposure[].amplitude_median` | El flujo de la primaria en ESA exposición, canal a canal. | Es el número que el combinado promedia; su dispersión mide cuánto varía la noche. |
| `subtraction.amplitude_median_spread_pct` | Recorrido de esas medianas. | Grande = transmisión/seeing muy variables entre exposiciones, que es justo el motivo de restar antes de combinar. |
| `combine.rejected_fraction` | Vóxeles que el sigma-clip tira. | Es lo único que rompe la equivalencia entre restar-luego-combinar y combinar-luego-restar. |


## Resultados que llevaron a la conclusión

Con `method="mean"` restar-luego-combinar es **exactamente** combinar-luego-restar —la combinación es lineal—, y eso está fijado por test. Con `sigclip` no lo es: la diferencia es el recorte, y se publica en `combine.rejected_fraction` en vez de suponerse despreciable.


## Decisiones y notas
- El orden es: modelo por exposición → resta → combinado. Restar sobre el combinado obliga a describir con un solo modelo la mezcla de 29–30 PSF distintas. · [`docs/2026-08-15_psf_por_observacion.md`](../docs/2026-08-15_psf_por_observacion.md)
- El fondo ajustado NO se resta (igual que en C3) y `STAT` no se toca: restar un modelo determinista no cambia la varianza.
- Sólo la variante `psfsub` de C3 consume este cubo; C2, C4, C5 y C6 siguen sobre el combinado de B2. · [`docs/spec_C1b_codex_perobs_subtraction.md`](../docs/spec_C1b_codex_perobs_subtraction.md)
